### Extracting data from a webpage using WebBaseLoader

In [52]:
from langchain_community.document_loaders import WebBaseLoader

URL = "https://www.geeksforgeeks.org/system-design/what-is-a-modular-monolith/"

loader = WebBaseLoader(URL)

doc = loader.load()

document = {
    "metadata": doc[0].metadata,
    "content": doc[0].page_content
}

document

{'metadata': {'source': 'https://www.geeksforgeeks.org/system-design/what-is-a-modular-monolith/',
  'title': 'What Is a Modular Monolith? - GeeksforGeeks',
  'description': 'Your All-in-One Learning Portal: GeeksforGeeks is a comprehensive educational platform that empowers learners across domains-spanning computer science and programming, school education, upskilling, commerce, software tools, competitive exams, and more.',
  'language': 'en'},
 'content': 'What Is a Modular Monolith? - GeeksforGeeksCoursesTutorialsInterview PrepSystem Design TutorialHLDLLDFunctional and Non FunctionalLife CycleDesign PatternsUML DiagramsSystem Design Interview GuideScalabilityDatabasesDSASoftware EngineeringWhat Is a Modular Monolith?Last Updated : 23 Jul, 2025In System Design, there are two main ways to structure big projects: the "all-in-one" approach called monolithic, and the "building block" approach called modular. But what if we could have the benefits of both? That\'s where the modular monol

### Splitting the webpage content into smaller chunks using RecursiveCharacterTextSplitter

In [53]:
from langchain_text_splitters import RecursiveCharacterTextSplitter  

raw_text = document["content"]

text_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)

texts = text_splitter.split_text(raw_text)

len(texts)

29

In [54]:
import re

# Tratar os splits que não terminam com um ponto final para que sejam concatenados ao split seguinte usando RE. 
treated_texts = []
for text in texts:
    if treated_texts and not re.search(r'[.!...]$', treated_texts[-1]):
        treated_texts[-1] += ' ' + text
    else:
        treated_texts.append(text)

len(treated_texts)

8

### Embedding the chunks using OpenAIEmbeddings and storing them in a Chroma vector store

In [62]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(
    model = "text-embedding-3-small", 
    dimensions= 1024
)

from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore.from_texts(treated_texts, embeddings)


### Retrieving relevant chunks based on a query using the retriever interface of the vector store

In [ ]:

retriever = vector_store.as_retriever()
retrieved_documents = retriever.invoke("What is a modular monolith?")
retrieved_documents[0].page_content

'What Is a Modular Monolith? - GeeksforGeeksCoursesTutorialsInterview PrepSystem Design TutorialHLDLLDFunctional and Non FunctionalLife CycleDesign PatternsUML DiagramsSystem Design Interview GuideScalabilityDatabasesDSASoftware EngineeringWhat Is a Modular Monolith?Last Updated : 23 Jul, 2025In System Design, there are two main ways to structure big projects: the "all-in-one" approach called monolithic, and the "building block" approach called modular. But what if we could have the benefits of both? That\'s where the modular monolith comes in.'